In [ ]:
# parameters
# BINDER_FAST: set N=6, T_ns=200, n_steps=40, n_iter=50 for fast cloud execution
N = 10              # Hilbert space truncation (Fock dimension)
omega0_GHz = 5.0    # bare cavity frequency (GHz)
K_GHz = 0.2         # Kerr nonlinearity (GHz)
T_ns = 500          # gate time (nanoseconds)
n_steps = 100       # time slices for the GRAPE propagator
n_iter = 200        # maximum GRAPE iterations
TWO_PI = 2 * 3.141592653589793

In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import snap_operator, snap_unitary_ideal, optimize_grape, GRAPEResult
from bosonic_gates.driven_kerr import DrivenKerrConfig, make_H0, make_operators

TWO_PI = 2 * np.pi

In [ ]:
# hide
from scipy.linalg import expm

def build_propagator_pwc(H0: qt.Qobj, H_ctrl: list, pulse_params: np.ndarray,
                         T: float, n_steps: int) -> qt.Qobj:
    """Build propagator by product of matrix exponentials for a PWC pulse.

    Parameters
    ----------
    H0 : qt.Qobj  drift Hamiltonian
    H_ctrl : list of qt.Qobj  control Hamiltonians
    pulse_params : np.ndarray, shape (n_ctrl, n_steps)
    T : float  total time
    n_steps : int

    Returns
    -------
    U : qt.Qobj  (dims [[N],[N]])
    """
    dim = H0.shape[0]
    dt = T / n_steps
    H0_np = H0.full()
    H_ctrl_np = [h.full() for h in H_ctrl]
    U = np.eye(dim, dtype=complex)
    for k in range(n_steps):
        H_k = H0_np.copy()
        for ci, Hc in enumerate(H_ctrl_np):
            H_k = H_k + pulse_params[ci, k] * Hc
        U = expm(-1j * H_k * dt) @ U
    U_qobj = qt.Qobj(U)
    U_qobj.dims = [[dim], [dim]]
    return U_qobj


def gate_fidelity(U_achieved: qt.Qobj, U_target: qt.Qobj) -> float:
    """Process fidelity F = |Tr(U†V)|² / d²."""
    d = U_target.shape[0]
    overlap = (U_achieved.dag() * U_target).tr()
    return float(np.abs(overlap)**2 / d**2)

## Module 5b: GRAPE Optimal Control for SNAP Gates

**Learning objectives:**
- Understand the GRAPE algorithm as gradient ascent on a physics simulation
- Set up the driven-Kerr system for optimal control: drift + control Hamiltonians
- Run `optimize_grape` from the `bosonic_gates` library and interpret the result
- Visualize convergence (infidelity history) and the optimal pulse shape
- Verify the optimized gate using direct propagator construction
- Assess robustness of the optimal pulse to control noise

---

**Sections:**
[1 GRAPE Algorithm](#sec1) ·
[2 Problem Setup](#sec2) ·
[3 Running GRAPE](#sec3) ·
[4 Optimal Pulse and Verification](#sec4) ·
[5 Robustness to Noise](#sec5) ·
[Exercises](#exercises)

<a id="sec1"></a>
## 1  The GRAPE Algorithm

**GRAPE** (GRadient Ascent Pulse Engineering) was introduced by Khaneja *et al.*
(*J. Magn. Reson.* **172**, 296, 2005) in the context of NMR pulse design.
It has since become the standard tool for quantum gate optimization in
superconducting circuits, trapped ions, and quantum optics.

### Objective

Find a piecewise-constant pulse $\{u_k\}_{k=1}^{n_{\rm steps}}$ that maximises

$$F(\{u_k\}) = \frac{|\mathrm{Tr}(\hat{U}^\dagger(\{u_k\})\,\hat{U}_{\rm target})|^2}{d^2}.$$

### Forward and backward propagation

Define the **forward propagators** starting from the identity:

$$P_0 = \mathbf{1}, \qquad P_k = e^{-i\hat{H}(t_k)\delta t}\,P_{k-1}.$$

The achieved gate is $\hat{U} = P_{n_{\rm steps}}$.  
Define the **co-state** (backward propagator weighted by target) at each slice:

$$Q_k = P_{k+1}^\dagger \cdots P_{n}^\dagger\,\hat{U}_{\rm target}.$$

### GRAPE gradient

The gradient of $F$ with respect to the $k$-th pulse amplitude is:

$$\frac{\partial F}{\partial u_k} =
  2\,\mathrm{Re}\!\left[\frac{\mathrm{Tr}(\hat{U}^\dagger \hat{U}_{\rm target})}{d^2}\cdot
  \mathrm{Tr}\!\left(Q_k^\dagger(-i\hat{H}_{\rm ctrl}\delta t)P_k\right)\right].$$

**Key insight:** computing $\{P_k\}$ (one forward pass) and $\{Q_k\}$ (one backward
pass) gives the gradient at *all* $n_{\rm steps}$ time slices simultaneously —
the same complexity as a single forward simulation.

### Update rule

$$u_k \leftarrow u_k + \alpha\,\frac{\partial F}{\partial u_k},$$

where $\alpha$ is the step size (learning rate), chosen adaptively by the optimizer
(Adam in the JAX backend, L-BFGS-B in the scipy fallback).

### JAX autodiff implementation

The `bosonic_gates` library uses **JAX automatic differentiation** through the
full propagator product.  Instead of implementing the forward/backward sweep
manually, JAX computes $\partial F/\partial u_k$ by differentiating through the
chain of Cayley approximants:

$$\hat{U}_k^{\rm Cayley} = \left(\mathbf{I} + \frac{i\hat{H}_k\delta t}{2}\right)^{-1}
\!\left(\mathbf{I} - \frac{i\hat{H}_k\delta t}{2}\right),$$

which is unitary to $\mathcal{O}(\delta t^2)$ and always invertible.  The
gradient is numerically equivalent to the analytic GRAPE formula but requires
no manual derivation.

---
*__Lab note.__ GRAPE has no convergence guarantee for the global optimum — the
fidelity landscape is non-convex and the algorithm may converge to a local
maximum.  In practice, multiple random initializations (different `seed` values)
and longer gate times $T$ both improve the final fidelity.*

<a id="sec2"></a>
## 2  Setting Up the Optimal Control Problem

The driven-Kerr model for a superconducting cavity is:

$$\hat{H}(t) = \underbrace{\omega_0\hat{n} - \frac{K}{2}\hat{n}(\hat{n}-1)}_{\hat{H}_0\;\text{(Kerr drift)}}
            + u(t)\underbrace{(\hat{a}+\hat{a}^\dagger)}_{\hat{H}_{\rm ctrl}\;\text{(X-quadrature drive)}}.$$

The Kerr nonlinearity $K$ is essential: it makes the transition frequencies
$\omega_{n,n+1} = \omega_0 - nK$ unequal, so a carefully shaped pulse can
selectively address individual Fock levels — the physical mechanism behind SNAP gates.

We target a **two-level SNAP gate** with $\theta_2 = \pi$ and $\theta_5 = \pi/2$:

$$\hat{U}_{\rm SNAP} = \sum_n e^{i\theta_n}|n\rangle\langle n|, \quad
\theta_2 = \pi, \quad \theta_5 = \pi/2, \quad \theta_{n\neq 2,5} = 0.$$

This is a non-trivial gate that requires the Kerr nonlinearity to resolve levels 2 and 5
independently — a flat pulse cannot do it.

In [ ]:
# System Hamiltonians
cfg = DrivenKerrConfig(N=N, omega0=TWO_PI * omega0_GHz * 1e9,
                       K=TWO_PI * K_GHz * 1e9)
H0 = make_H0(cfg)
a, adag, n_op = make_operators(N)
H_ctrl = [a + adag]   # single X-quadrature control channel

# Target SNAP gate
thetas = np.zeros(N)
thetas[2] = np.pi
thetas[5] = np.pi / 2
target_U = snap_unitary_ideal(N, thetas)

T = T_ns * 1e-9  # gate time in seconds

print("=== Optimal Control Problem Setup ===")
print(f"N = {N}  (Fock truncation)")
print(f"omega0/(2pi) = {cfg.omega0/TWO_PI/1e9:.3f} GHz")
print(f"K/(2pi)      = {cfg.K/TWO_PI/1e9:.3f} GHz")
print(f"Gate time T  = {T_ns} ns")
print(f"Time slices  = {n_steps}  (dt = {T_ns/n_steps:.2f} ns per slice)")
print(f"Pulse parameters: {len(H_ctrl) * n_steps}  (n_ctrl x n_steps)")
print(f"\nTarget SNAP phases (non-zero only):")
for n in range(N):
    if abs(thetas[n]) > 1e-10:
        print(f"  theta[{n}] = {thetas[n]:.4f} rad = {thetas[n]/np.pi:.3f} pi")

In [ ]:
# Visualise the target unitary
U_target_mat = target_U.full()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(np.abs(U_target_mat), origin='lower', cmap='viridis', vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0])
axes[0].set_title(r"$|U_{\rm target}|$")
axes[0].set_xlabel('$j$')
axes[0].set_ylabel('$i$')

im1 = axes[1].imshow(np.angle(U_target_mat), origin='lower', cmap='hsv',
                     vmin=-np.pi, vmax=np.pi)
plt.colorbar(im1, ax=axes[1], label='phase (rad)')
axes[1].set_title(r"$\arg(U_{\rm target})$ — SNAP phases")
axes[1].set_xlabel('$j$')
axes[1].set_ylabel('$i$')

plt.suptitle(
    rf"Target SNAP gate: $\theta_2=\pi$, $\theta_5=\pi/2$", fontsize=12)
plt.tight_layout()
plt.show()

# Show effect on a superposition state
print("\nEffect on uniform superposition (1/sqrt(N)) sum_n |n>):")
psi_super = qt.Qobj(np.ones(N) / np.sqrt(N))
psi_super.dims = [[N], [1]]
psi_out = target_U * psi_super
print("  n    |<n|psi>|     phase")
for n in range(N):
    amp = float(abs(psi_out[n][0][0]))
    ph  = float(np.angle(psi_out[n][0][0]))
    print(f"  {n}    {amp:.4f}       {ph:.4f} rad")

<a id="sec3"></a>
## 3  Running GRAPE

We call `optimize_grape` with the target unitary, drift and control Hamiltonians,
gate time $T$, and optimization parameters.  The function returns a `GRAPEResult`
containing the optimal pulse, fidelity history, and convergence status.

**Initialization:** pulses start as small random perturbations around zero
(controlled by `seed`).  Different seeds explore different valleys in the
fidelity landscape.

**Backend:** `use_jax=True` selects the JAX+optax Adam optimizer (fast, GPU-ready);
`use_jax=False` falls back to scipy L-BFGS-B with finite-difference gradients
(slower but always available).

In [ ]:
# Run GRAPE
result = optimize_grape(
    target_U, H0, H_ctrl,
    T=T, n_steps=n_steps,
    n_iter=n_iter,
    seed=42,
    use_jax=True,
)

print(result)
print(f"\nFinal fidelity:   F = {result.fidelity:.6f}")
print(f"Final infidelity: 1-F = {result.infidelity:.3e}")
print(f"Converged:        {result.converged}")
print(f"Method:           {result.method}")
print(f"Iterations run:   {len(result.fidelity_history)}")
if result.fidelity_history:
    print(f"Initial fidelity: {result.fidelity_history[0]:.6f}")
    print(f"Improvement:      {result.fidelity - result.fidelity_history[0]:.6f}")

In [ ]:
# Plot infidelity convergence on semi-log axis
infid_history = [1.0 - F for F in result.fidelity_history]
iters = np.arange(1, len(infid_history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: fidelity (linear)
axes[0].plot(iters, result.fidelity_history, lw=2, color='steelblue')
axes[0].axhline(1.0, color='gray', ls='--', lw=1)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel(r'Gate fidelity $F$')
axes[0].set_title('GRAPE convergence (fidelity)')
axes[0].set_ylim(-0.05, 1.05)
axes[0].annotate(
    rf'$F_{{\rm final}} = {result.fidelity:.4f}$',
    xy=(iters[-1], result.fidelity),
    xytext=(0.55, 0.3), textcoords='axes fraction',
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10,
)

# Right: infidelity (semi-log)
axes[1].semilogy(iters, infid_history, lw=2, color='darkorange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel(r'Infidelity $1-F$')
axes[1].set_title('GRAPE convergence (infidelity, semi-log)')
axes[1].annotate(
    rf'$1-F_{{\rm final}} = {result.infidelity:.2e}$',
    xy=(iters[-1], result.infidelity),
    xytext=(0.4, 0.6), textcoords='axes fraction',
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=10,
)

plt.suptitle(
    rf"GRAPE convergence: SNAP gate $N={N}$, $T={T_ns}$ ns, {n_steps} time slices",
    fontsize=11)
plt.tight_layout()
plt.show()

**Observation.** The infidelity decreases rapidly in the first few tens of iterations
(fast gradient-driven descent) and then slows as the optimizer approaches a local optimum.
The semi-log plot reveals the asymptotic rate of convergence.

---
*__Lab note.__ The initial fidelity from a random seed near zero is $\approx 1/N^2$
— the random unitary average.  The fact that GRAPE can drive this to $F > 0.99$
in $\sim 200$ iterations demonstrates the power of gradient information.
Without gradients, a random-search optimizer would need $\sim N^2$ function
evaluations per unit of improvement, which is prohibitive for large $N$.*

<a id="sec4"></a>
## 4  Optimal Pulse Shape and Gate Verification

After optimization we have the optimal pulse $\{u_k^*\}$ stored in
`result.pulse_params`.  We:
1. Plot the pulse as a step function (piecewise constant in time)
2. Reconstruct the propagator from the pulse using the exact product-of-exponentials formula
3. Verify the gate fidelity matches the optimizer's reported value
4. Compare the Wigner function of the output state to the ideal SNAP output

In [ ]:
# Plot the optimal pulse
tlist_ns = np.linspace(0, T_ns, n_steps)  # nanoseconds
optimal_pulse = result.pulse_params[0]     # first (only) control channel

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.step(tlist_ns, optimal_pulse, where='post', lw=1.5, color='steelblue')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Pulse amplitude $u_k$ (rad/s)')
ax.set_title(
    rf"Optimal GRAPE pulse: SNAP $\theta_2=\pi$, $\theta_5=\pi/2$,"
    rf" $T={T_ns}$ ns, $F={result.fidelity:.4f}$")
plt.tight_layout()
plt.show()

print(f"Pulse statistics:")
print(f"  Mean amplitude:     {np.mean(optimal_pulse):.4f} rad/s")
print(f"  RMS amplitude:      {np.sqrt(np.mean(optimal_pulse**2)):.4f} rad/s")
print(f"  Max amplitude:      {np.max(np.abs(optimal_pulse)):.4f} rad/s")
print(f"  Total pulse area:   {np.sum(optimal_pulse) * T/n_steps:.4f} rad")

In [ ]:
# Verify: reconstruct propagator and compute fidelity independently
U_optimal = build_propagator_pwc(H0, H_ctrl, result.pulse_params, T, n_steps)
F_verify  = gate_fidelity(U_optimal, target_U)

print(f"Verification:")
print(f"  GRAPE reported fidelity:   F = {result.fidelity:.6f}")
print(f"  Independently computed F:  F = {F_verify:.6f}")
print(f"  Difference:                   {abs(F_verify - result.fidelity):.2e}")

# Phase of each Fock component under the optimal gate
print("\nDiagonal of optimal propagator (phase applied to each |n>):")
U_opt_mat = U_optimal.full()
print(f"  {'n':>3}  {'|U_nn|':>8}  {'arg(U_nn)/pi':>14}  {'target':>10}")
for n in range(N):
    amp_nn  = float(abs(U_opt_mat[n, n]))
    phase_nn = float(np.angle(U_opt_mat[n, n])) / np.pi
    target_phase = thetas[n] / np.pi
    print(f"  {n:3d}  {amp_nn:8.4f}  {phase_nn:14.4f} pi  {target_phase:10.4f} pi")

In [ ]:
# Wigner function comparison: ideal SNAP vs GRAPE on initial |0>
psi0 = qt.basis(N, 0)

psi_ideal = target_U * psi0
psi_grape = U_optimal * psi0
psi_grape = psi_grape.unit()

xvec = np.linspace(-4, 4, 100)
W_ideal = qt.wigner(qt.ket2dm(psi_ideal), xvec, xvec)
W_grape = qt.wigner(qt.ket2dm(psi_grape), xvec, xvec)
W_diff  = W_ideal - W_grape

vmax = max(np.max(np.abs(W_ideal)), np.max(np.abs(W_grape)))
vmax_d = np.max(np.abs(W_diff))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, W, title, vm, cmap in zip(
    axes,
    [W_ideal, W_grape, W_diff],
    [r"(a) Ideal SNAP $|0\rangle$",
     rf"(b) GRAPE ($F={result.fidelity:.4f}$) $|0\rangle$",
     r"(c) Difference"],
    [vmax, vmax, vmax_d],
    ['RdBu_r', 'RdBu_r', 'PiYG'],
):
    ax.contourf(xvec, xvec, W, levels=60, cmap=cmap, vmin=-vm, vmax=vm)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r'Re$(\alpha)$')
    ax.set_ylabel(r'Im$(\alpha)$')
    ax.set_aspect('equal')

plt.suptitle(
    rf"Wigner functions after SNAP gate on $|0\rangle$: ideal vs GRAPE",
    fontsize=11)
plt.tight_layout()
plt.show()

state_fid = float(abs(psi_ideal.dag() * psi_grape)**2)
print(f"State fidelity |<psi_ideal|psi_grape>|^2 = {state_fid:.6f}")
print(f"Max |W_diff| = {vmax_d:.4f}  (0 = perfect match)")

**Observation.** Panel (c) shows the Wigner-function residual.  For high-fidelity
gates ($F > 0.99$), the difference is small: the two states are nearly identical in
phase space.  Residual structure in panel (c) indicates where the GRAPE gate
deviates from ideal — typically at the Wigner fringes associated with the
phase-flipped Fock levels.

---
*__Lab note.__ The GRAPE-optimized pulse is a sequence of microwave amplitudes
uploaded to an AWG.  In the lab, the pulse is filtered by the AWG bandwidth and
the cavity's input coupling.  Robust-optimal-control techniques (e.g., adding
a penalty on pulse bandwidth) can mitigate this distortion during optimization.*

<a id="sec5"></a>
## 5  Gate Robustness to Pulse Noise

In experiment, the realized pulse $\{u_k\}$ deviates from the ideal due to
finite AWG precision, amplifier noise, and cable reflections.  We model this as
additive Gaussian noise:

$$u_k^{\rm noisy} = u_k^* + \sigma\,\xi_k, \qquad \xi_k \sim \mathcal{N}(0, 1).$$

The resulting fidelity degradation characterizes the **robustness** of the
GRAPE solution.  Pulses with large RMS amplitudes tend to be less robust because
small perturbations produce large phase errors.

Experimentally, robustness is improved by adding a **bandwidth penalty** or
**amplitude regularization** to the GRAPE objective, or by applying a DRAG-like
smoothing filter to the optimal pulse before uploading to the AWG.

In [ ]:
# Fidelity vs noise level
sigma_vals = np.array([0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2])
n_trials = 20   # average over multiple noise realizations
rng = np.random.default_rng(0)

F_mean = []
F_std  = []

for sigma in sigma_vals:
    fids = []
    for _ in range(n_trials):
        noise = rng.standard_normal(result.pulse_params.shape)
        noisy_params = result.pulse_params + sigma * noise
        U_noisy = build_propagator_pwc(H0, H_ctrl, noisy_params, T, n_steps)
        fids.append(gate_fidelity(U_noisy, target_U))
    F_mean.append(np.mean(fids))
    F_std.append(np.std(fids))
    print(f"  sigma = {sigma:.3f}  ->  F = {F_mean[-1]:.4f} +/- {F_std[-1]:.4f}")

F_mean = np.array(F_mean)
F_std  = np.array(F_std)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(sigma_vals, F_mean, yerr=F_std, fmt='o-', lw=2, capsize=4,
            color='steelblue', ecolor='steelblue', alpha=0.8)
ax.axhline(result.fidelity, color='gray', ls='--', lw=1,
           label=rf"Noise-free $F = {result.fidelity:.4f}$")
ax.set_xlabel(r'Noise amplitude $\sigma$ (relative to pulse RMS)')
ax.set_ylabel(r'Gate fidelity $F$')
ax.set_title(rf'GRAPE gate robustness: $F$ vs pulse noise (N={N}, T={T_ns} ns)')
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

# Find 1% infidelity noise floor
idx_1pct = np.searchsorted(1.0 - F_mean, 0.01)
if idx_1pct < len(sigma_vals):
    print(f"\nFidelity drops below 0.99 at sigma ~ {sigma_vals[min(idx_1pct, len(sigma_vals)-1)]:.3f}")
else:
    print("\nFidelity stays > 0.99 across all tested noise levels")

**Observation.** The GRAPE gate is highly robust for small noise ($\sigma \ll 0.01$)
and degrades gracefully at larger amplitudes.  The error bars (standard deviation
across noise realizations) show the variability: a single shot with high noise can
produce fidelity much worse than the mean.

---
*__Lab note.__ Experimental control noise in superconducting circuits is typically
$\sigma \lesssim 1\%$ of the pulse amplitude (AWG 10-bit precision, amplifier noise
figure $\sim 3$ dB).  GRAPE pulses optimized with a smoothness penalty are more
robust and also easier to upload: fewer harmonics means the AWG filter distorts them less.*

<a id="exercises"></a>
## Exercises

**Exercise 1.** Run GRAPE with only `n_iter=20` iterations (underpowered) —
what is the final fidelity?  Plot the convergence curve alongside the full
`n_iter` run.  At what iteration does most of the fidelity gain occur?

**Exercise 2.** Try a different target: single-level SNAP with only
`thetas[3] = np.pi` (all other levels unaffected).  Does GRAPE converge faster
compared to the two-level target?

In [ ]:
# Exercise 1 — underpowered GRAPE
# YOUR CODE HERE

In [ ]:
# Exercise 2 — single-level SNAP target
# YOUR CODE HERE

In [ ]:
# Solution — Exercise 1: underpowered GRAPE
result_short = optimize_grape(
    target_U, H0, H_ctrl, T=T, n_steps=n_steps,
    n_iter=20, seed=42, use_jax=True,
)

print(f"Short run (n_iter=20):   F = {result_short.fidelity:.6f}")
print(f"Full run  (n_iter={n_iter}): F = {result.fidelity:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(result.fidelity_history, lw=2, label=rf'n_iter={n_iter}')
ax.plot(result_short.fidelity_history, lw=2, ls='--', label='n_iter=20')
ax.axvline(20, color='gray', ls=':', lw=1)
ax.set_xlabel('Iteration')
ax.set_ylabel('Fidelity $F$')
ax.set_title('GRAPE convergence: full vs underpowered')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Solution — Exercise 2: single-level SNAP
thetas_single = np.zeros(N)
thetas_single[3] = np.pi
target_single = snap_unitary_ideal(N, thetas_single)

result_single = optimize_grape(
    target_single, H0, H_ctrl, T=T, n_steps=n_steps,
    n_iter=n_iter, seed=42, use_jax=True,
)

print(f"Single-level SNAP (theta[3]=pi):  F = {result_single.fidelity:.6f}")
print(f"Two-level SNAP    (theta[2,5]):    F = {result.fidelity:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy([1-f for f in result.fidelity_history], lw=2,
            label=r'Two-level SNAP ($\theta_2=\pi$, $\theta_5=\pi/2$)')
ax.semilogy([1-f for f in result_single.fidelity_history], lw=2, ls='--',
            label=r'Single-level SNAP ($\theta_3=\pi$)')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'Infidelity $1-F$')
ax.set_title('GRAPE convergence: two-level vs single-level target')
ax.legend()
plt.tight_layout()
plt.show()